# text2sql-lora — QLoRA fine-tuning on a free Colab T4

End-to-end GPU phase of the [SQLoRA](https://github.com/BenBrahimMazen/SQLora) pipeline, on real Spider data:

1. **Smoke tests** — 20-question inference and a 10-step training run, to verify the whole stack before committing hours.
2. **Full runs** — zero-shot baseline, few-shot baseline, QLoRA fine-tuning, fine-tuned predictions.
3. **Evaluation** — exact match + execution accuracy against the real Spider databases, CSV + chart.
4. **Artifact download** — adapter, logs, predictions, and results zipped back to your machine.

**Before anything:** `Runtime > Change runtime type > T4 GPU`.

| Stage | Approx. time on T4 |
|---|---|
| Setup (clone, installs, data download) | ~5–10 min |
| Smoke tests | ~10–15 min |
| Zero-shot baseline (1,034 dev questions) | ~30–45 min |
| Few-shot baseline (k=3) | ~1–1.5 h |
| QLoRA training (3 epochs, 7,000 examples) | ~2 h |
| Fine-tuned predictions | ~30–45 min |

Every cell writes its output to disk (`preds/`, `outputs/`, `results/`), so if the session drops you can re-run the setup cell and jump straight back to where you were — cells whose output file already exists can be skipped. Keep this tab open during long runs.

In [ ]:
# GPU check — stop here if this fails: Runtime > Change runtime type > T4 GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then re-run this cell'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Clone the repo and install the pinned stack (idempotent — safe to re-run after a restart)
import os
if not os.path.isdir('SQLora'):
    !git clone https://github.com/BenBrahimMazen/SQLora.git
%cd SQLora
%pip install -q -r requirements.txt

import torch, transformers, peft, trl, bitsandbytes, datasets
for m in (torch, transformers, peft, trl, bitsandbytes, datasets):
    print(f'{m.__name__:<15} {m.__version__}')

In [ ]:
# Real Spider data: ~210 MB download (questions + gold SQL + 166 SQLite databases), then preprocess
!python scripts/download_spider.py
!python src/preprocessing.py

## 1 — Smoke tests

Verify the full stack on tiny slices first: 20 dev questions through the base model, then 10 optimizer steps of QLoRA. If both cells finish cleanly, the long runs below are safe to start. First inference cell also downloads the model weights (~6 GB), so expect a few extra minutes on its first run.

In [ ]:
# Inference smoke test: 20 dev questions, zero-shot (~5 min + model download)
!python src/baseline_prompting.py --limit 20 --out preds/smoke_baseline.jsonl

In [ ]:
# Training smoke test: 10 optimizer steps end-to-end (load 4-bit, train, save adapter)
!python src/train_qlora.py --max-steps 10 --output-dir outputs/smoke_run

## 2 — Full runs

Three prediction files and one adapter, in dependency order. Each cell is independent once the previous one's output file exists, so a dropped session only costs the cell that was running. Training logs loss every 10 steps and writes the full history to `outputs/qlora_run/log_history.json`.

In [ ]:
# Zero-shot baseline: all 1,034 dev questions, greedy decoding (~30-45 min)
!python src/baseline_prompting.py --out preds/baseline_zeroshot.jsonl

In [ ]:
# Few-shot baseline: 3 fixed exemplars from train, reused for every question (~1-1.5 h)
!python src/baseline_prompting.py --few-shot-k 3 --out preds/baseline_fewshot.jsonl

In [ ]:
# QLoRA fine-tuning: 3 epochs, ~2 h on a T4. Keep this tab open while it runs.
!python src/train_qlora.py --config configs/default.yaml

In [ ]:
# Fine-tuned predictions: same inference script, pointed at the trained adapter
!python src/baseline_prompting.py --adapter outputs/qlora_run/final_adapter --out preds/qlora.jsonl

## 3 — Evaluate

Exact match + execution accuracy (predicted vs gold result sets, in the read-only sandbox), overall and per difficulty tier. Writes `results/summary.csv` (same columns as the README Results table) and `results/accuracy_by_difficulty.png`.

In [ ]:
!python src/evaluate.py --pred baseline-zeroshot=preds/baseline_zeroshot.jsonl --pred baseline-fewshot=preds/baseline_fewshot.jsonl --pred qlora=preds/qlora.jsonl

import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('results/summary.csv'))
display(Image('results/accuracy_by_difficulty.png'))

## 4 — Take the artifacts home

Colab disks are ephemeral: everything needed to fill the README Results table and to keep the fine-tuned model lives in this zip — adapter weights, resolved config, loss history, predictions, and results.

In [ ]:
!zip -qr qlora_artifacts.zip outputs/qlora_run/final_adapter outputs/qlora_run/log_history.json outputs/qlora_run/resolved_config.yaml preds results
print('zip size (MB):', round(os.path.getsize('qlora_artifacts.zip') / 1e6))

from google.colab import files
files.download('qlora_artifacts.zip')

# If the browser download is flaky for a large zip, back it up to Drive instead:
# from google.colab import drive; drive.mount('/content/drive')
# !cp qlora_artifacts.zip /content/drive/MyDrive/

## Optional — GGUF export for CPU inference

Only after a successful run: merge the adapter into the base model and export a quantized GGUF for the llama.cpp demo path. Clones `llama.cpp` (~150 MB) and needs `convert_hf_to_gguf.py`'s Python deps.

In [ ]:
# Optional: merge + GGUF (~20-30 min, CPU)
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
%pip install -q gguf
!python src/merge_and_quantize.py --adapter outputs/qlora_run/final_adapter --gguf

---

**Back on your machine:** unzip into the repo root, then fill the README Results table strictly from `results/summary.csv` — real numbers only — and commit `results/` as evidence of the runs.